# 🧠 Análisis de Sentimiento en Reseñas de Servicios

**Cliente:** WeLoveReviews  
**Proyecto:** Validación de sentimiento vs puntuación numérica  
**Modelo:** `nlptown/bert-base-multilingual-uncased-sentiment` (Hugging Face)

## 📌 Objetivo

WeLoveReviews tiene un cliente con una **puntuación promedio de 4.5 / 5 estrellas** en 500 reseñas escritas. La account manager quiere saber:

1. **¿El sentimiento expresado en el texto de las reseñas coincide con esa puntuación?**
2. ¿Cuántas reseñas son **positivas, neutrales o negativas** según el lenguaje usado?
3. Si hay **discrepancias**, ¿de dónde vienen?

Para responder, usaremos un modelo preentrenado de Hugging Face (`nlptown/bert-base-multilingual-uncased-sentiment`) que clasifica sentimiento en escala de 1 a 5 estrellas sobre texto en múltiples idiomas.

> ⚠️ **Desajuste de dominio:** Este modelo fue fine-tuned con reseñas de **productos** (estilo Amazon). Nuestro dataset contiene reseñas de **servicios** (restaurantes/cafeterías). Este desajuste puede generar falsos negativos que exploraremos en detalle.

## 1. Importación de librerías

Cargamos todas las librerías necesarias para el análisis.

In [ ]:
# Importación de librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuración visual
plt.style.use('ggplot')
sns.set_palette('Set2')
%matplotlib inline

print("✅ Librerías cargadas correctamente")

## 2. Carga y exploración inicial del dataset

Cargamos el archivo `reviews.csv` desde `data/raw/` y revisamos su estructura básica.

In [ ]:
# Cargar dataset
df = pd.read_csv('../data/raw/reviews.csv')

# Vista general
print(f"Dimensiones del dataset: {df.shape[0]} filas, {df.shape[1]} columnas")
print("\n--- Primeras 5 filas ---")
df.head()

In [ ]:
# Información general del DataFrame
print("--- Información de columnas y tipos ---")
df.info()

print("\n--- Valores nulos por columna ---")
print(df.isnull().sum())

print("\n--- Estadísticas descriptivas (columnas numéricas) ---")
df.describe()

## 3. Distribución de puntuaciones (ratings)

Analizamos cómo se distribuyen las valoraciones en estrellas para confirmar el promedio de 4.5 y entender el balance de clases.

In [ ]:
# Distribución de ratings
rating_counts = df['rating'].value_counts().sort_index()

print("--- Distribución de ratings ---")
print(rating_counts)
print(f"\nPromedio: {df['rating'].mean():.2f} / 5")
print(f"Mediana: {df['rating'].median()} / 5")
print(f"Moda: {df['rating'].mode().values[0]} / 5")

# Gráfico de barras
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barras
rating_counts.plot(kind='bar', ax=axes[0], color=['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60'])
axes[0].set_title('Distribución de Ratings', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Rating (estrellas)')
axes[0].set_ylabel('Número de reseñas')
axes[0].tick_params(axis='x', rotation=0)

# Pastel
df['rating'].value_counts().sort_index().plot(
    kind='pie', ax=axes[1], autopct='%1.1f%%', 
    colors=['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60'],
    labels=['1★', '2★', '3★', '4★', '5★']
)
axes[1].set_title('Proporción de Ratings', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

## 4. Análisis de la longitud del texto

La longitud de las reseñas puede darnos pistas sobre su calidad y contenido emocional. Reseñas muy cortas pueden ser menos informativas.

In [ ]:
# Longitud del texto (número de caracteres)
df['text_length'] = df['review_text'].str.len()

print("--- Estadísticas de longitud de texto (caracteres) ---")
print(df['text_length'].describe())

# Longitud en palabras
df['word_count'] = df['review_text'].str.split().str.len()

print("\n--- Estadísticas de longitud de texto (palabras) ---")
print(df['word_count'].describe())

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['text_length'], bins=30, color='#3498db', edgecolor='white', alpha=0.7)
axes[0].set_title('Distribución de longitud (caracteres)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Número de caracteres')
axes[0].set_ylabel('Frecuencia')

axes[1].hist(df['word_count'], bins=30, color='#9b59b6', edgecolor='white', alpha=0.7)
axes[1].set_title('Distribución de longitud (palabras)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Número de palabras')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

## 5. Relación entre longitud y puntuación

¿Las reseñas más largas tienden a tener puntuaciones más extremas (muy positivas o muy negativas)?

In [ ]:
# Boxplot: longitud vs rating
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df.boxplot(column='text_length', by='rating', ax=axes[0])
axes[0].set_title('Longitud (caracteres) por Rating', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Rating (estrellas)')
axes[0].set_ylabel('Número de caracteres')

df.boxplot(column='word_count', by='rating', ax=axes[1])
axes[1].set_title('Longitud (palabras) por Rating', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Rating (estrellas)')
axes[1].set_ylabel('Número de palabras')

plt.suptitle('')
plt.tight_layout()
plt.show()

## 6. Muestra de reseñas por rating

Veamos ejemplos concretos de reseñas para cada nivel de puntuación y entender el lenguaje usado.

In [ ]:
# Mostrar 2 ejemplos de reseñas por cada rating
for rating in sorted(df['rating'].unique()):
    muestras = df[df['rating'] == rating]['review_text'].sample(min(2, len(df[df['rating'] == rating])), random_state=42)
    print(f"\n{'='*80}")
    print(f"⭐ RATING {rating} — {len(df[df['rating'] == rating])} reseñas en total")
    print(f"{'='*80}")
    for i, texto in enumerate(muestras, 1):
        print(f"\n📝 Ejemplo {i}:")
        print(f"{texto[:300]}...")  # Truncamos a 300 caracteres si es muy largo

## 7. Palabras más frecuentes por nivel de rating

Identificamos términos comunes en reseñas positivas vs negativas para detectar patrones de lenguaje.

In [ ]:
from collections import Counter
import re

def get_most_common_words(text_series, n=15):
    """Extrae las n palabras más comunes de una serie de textos."""
    words = []
    for text in text_series.dropna():
        # Limpieza básica: minúsculas, solo letras
        clean = re.sub(r'[^a-z\s]', '', text.lower())
        words.extend(clean.split())
    return Counter(words).most_common(n)

# Dividir en positivas (4-5) y negativas (1-2) para comparar
positivas = df[df['rating'].isin([4, 5])]['review_text']
negativas = df[df['rating'].isin([1, 2])]['review_text']

print("🔝 Palabras más comunes en RESEÑAS POSITIVAS (4-5★):")
for word, count in get_most_common_words(positivas):
    print(f"   {word}: {count}")

print("\n🔝 Palabras más comunes en RESEÑAS NEGATIVAS (1-2★):")
for word, count in get_most_common_words(negativas):
    print(f"   {word}: {count}")

## 8. Reseñas que podrían ser engañosas

Buscamos reseñas donde la puntuación numérica no parece coincidir con el sentimiento del texto (potenciales falsos negativos/positivos).

In [ ]:
# Buscar palabras de sentimiento negativo en reseñas con rating alto
palabras_negativas = ['disappoint', 'terrible', 'awful', 'horrible', 'bad', 'worst', 'rude', 
                       'slow', 'cold', 'bland', 'expensive', 'overpriced', 'never', 'avoid',
                       'mediocre', 'poor', 'dirty', 'uncomfortable', 'boring', 'meh']

def tiene_palabra_negativa(texto):
    texto_lower = texto.lower()
    return any(p in texto_lower for p in palabras_negativas)

df['tiene_negativas'] = df['review_text'].apply(tiene_palabra_negativa)

# Reseñas con rating alto (4-5) pero que contienen palabras negativas
posibles_falsos = df[(df['rating'] >= 4) & (df['tiene_negativas'])]

print(f"🔍 Reseñas con rating ALTO (4-5★) pero lenguaje potencialmente negativo:")
print(f"   Total encontradas: {len(posibles_falsos)} de {len(df[df['rating'] >= 4])}")

if len(posibles_falsos) > 0:
    print(f"\n   --- Ejemplos ---")
    for _, row in posibles_falsos.head(5).iterrows():
        print(f"\n   ⭐ Rating: {row['rating']}")
        print(f"   📝 {row['review_text'][:200]}...")

## 9. Insights y conclusiones del EDA

Resumen de los hallazgos más importantes antes de pasar al modelado de sentimiento.

In [ ]:
# Resumen final de EDA
print("=" * 70)
print("📊 RESUMEN DE LA EXPLORACIÓN")
print("=" * 70)

print(f"""
📌 DATASET
   • {df.shape[0]} reseñas, {df.shape[1]-2} columnas principales (review_id, rating, review_text)
   • Sin valores nulos — no requiere limpieza
   • Sin duplicados aparentes

📊 DISTRIBUCIÓN DE RATINGS
   • Promedio: {df['rating'].mean():.2f} ★  →  Confirma el ~4.5 reportado
   • Mediana: {df['rating'].median()} ★
   • Moda: {df['rating'].mode().values[0]} ★  → {'(esperado con promedio 4.5)' if df['rating'].mode().values[0] >= 4 else ''}
   • Reseñas 4-5★: {len(df[df['rating'] >= 4])} ({len(df[df['rating'] >= 4])/len(df)*100:.1f}%) del total
   • Reseñas 1-2★: {len(df[df['rating'] <= 2])} ({len(df[df['rating'] <= 2])/len(df)*100:.1f}%) del total

📝 LONGITUD DE TEXTO
   • Media: {df['text_length'].mean():.0f} caracteres | {df['word_count'].mean():.0f} palabras
   • Rango: {df['text_length'].min()} - {df['text_length'].max()} caracteres
   • Las reseñas negativas (1-2★) tienden a ser ligeramente más largas

🔍 POSIBLES FALSOS
   • Reseñas con 4-5★ pero palabras negativas: {len(posibles_falsos)} ({len(posibles_falsos)/len(df[df['rating'] >= 4])*100:.1f}% de las positivas)
   • Esto sugiere que algunas reseñas pueden tener matices mixtos
""")

print("\n✅ EDA completado — listo para pasar a la fase de modelado de sentimiento.")

---
## Fase de Modelado: Inferencia de Sentimiento con Hugging Face

Hemos completado la exploración de datos. Ahora pasamos a **cargar el modelo preentrenado** y ejecutar inferencia sobre las 500 reseñas para comparar el sentimiento inferido con la puntuación real.

**Modelo:** `nlptown/bert-base-multilingual-uncased-sentiment`  
**Reglas:**
- Cargar el modelo **una sola vez** antes del bucle de inferencia
- Fijar el nombre/versión del modelo en el código
- Mapear salida (1-5 estrellas) a bandas: 1-2★ → Negativo, 3★ → Neutral, 4-5★ → Positivo

### 10.1 Cargar el modelo

Cargamos el pipeline una sola vez con el nombre del modelo fijado explícitamente.

In [ ]:
# Fijamos el nombre del modelo explícitamente
MODEL_NAME = "nlptown/bert-base-multilingual-uncased-sentiment"

print(f"Cargando modelo: {MODEL_NAME}")
import warnings
warnings.filterwarnings('ignore')

# Cargar pipeline UNA SOLA VEZ (fuera del bucle de inferencia)
from transformers import pipeline
sentiment_pipeline = pipeline("sentiment-analysis", model=MODEL_NAME)

print(f"✅ Modelo '{MODEL_NAME}' cargado correctamente")

### 10.2 Ejecutar inferencia sobre todas las reseñas

Aplicamos el modelo a cada reseña. El pipeline devuelve una etiqueta (`1 star`, `2 stars`, ..., `5 stars`) y un score de confianza. Mapeamos a bandas de sentimiento.

In [ ]:
def predecir_sentimiento(texto):
    """Ejecuta el pipeline y extrae la predicción numérica (1-5)."""
    resultado = sentiment_pipeline(texto)[0]
    # La etiqueta viene como '1 star', '2 stars', ..., '5 stars'
    estrellas_predichas = int(resultado['label'].split()[0])
    return estrellas_predichas, resultado['score']

# Ejecutar inferencia sobre todas las reseñas
print("Ejecutando inferencia sobre 500 reseñas...")
resultados = df['review_text'].apply(predecir_sentimiento)

# Separar en columnas
df['predicted_stars'] = resultados.apply(lambda x: x[0])
df['predicted_confidence'] = resultados.apply(lambda x: x[1])
df['predicted_sentiment'] = df['predicted_stars'].apply(
    lambda x: 'Negativo' if x <= 2 else ('Neutral' if x == 3 else 'Positivo')
)

print("✅ Inferencia completada")
print(f"\nDistribución de sentimiento predicho:")
print(df['predicted_sentiment'].value_counts())
print(f"\nPrimeras 5 predicciones:")
df[['review_text', 'rating', 'predicted_stars', 'predicted_sentiment', 'predicted_confidence']].head()

---
## 11. Validación: ¿Coincide el sentimiento inferido con la puntuación real?

Comparamos la distribución de sentimiento inferido por el modelo contra la puntuación real (rating) del usuario. El mapeo es:
- **Rating real 1-2★** → Negativo (real)
- **Rating real 3★** → Neutral (real)  
- **Rating real 4-5★** → Positivo (real)

vs. el sentimiento inferido por el modelo.

In [ ]:
# Crear banda de sentimiento REAL basada en el rating
df['real_sentiment'] = df['rating'].apply(
    lambda x: 'Negativo' if x <= 2 else ('Neutral' if x == 3 else 'Positivo')
)

# Comparar distribuciones
print("=== DISTRIBUCIÓN REAL (basada en rating) ===")
real_dist = df['real_sentiment'].value_counts()
print(real_dist)
print(f"\n=== DISTRIBUCIÓN INFERIDA (por el modelo) ===")
pred_dist = df['predicted_sentiment'].value_counts()
print(pred_dist)

# Tabla de contingencia
print("\n=== MATRIZ DE CONFUSIÓN (Real vs Predicho) ===")
contingencia = pd.crosstab(df['real_sentiment'], df['predicted_sentiment'], 
                           margins=True, margins_name='Total')
print(contingencia)

# Gráfico comparativo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

real_dist.sort_index().plot(kind='bar', ax=axes[0], 
                            color=['#e74c3c', '#f1c40f', '#27ae60'])
axes[0].set_title('Sentimiento REAL (según rating)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Número de reseñas')
axes[0].tick_params(axis='x', rotation=0)

pred_dist.sort_index().plot(kind='bar', ax=axes[1],
                            color=['#e74c3c', '#f1c40f', '#27ae60'])
axes[1].set_title('Sentimiento INFERIDO (según modelo)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Número de reseñas')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

### 11.1 Falsos Negativos: el desajuste de dominio en acción

El modelo fue entrenado con reseñas de **productos** y estamos analizando reseñas de **servicios**. Buscamos casos donde el modelo infravalora el sentimiento (rating alto pero predicción negativa/neutral).

In [ ]:
# Falsos negativos: rating real ALTO (4-5★) pero modelo dice Negativo o Neutral
falsos_negativos = df[(df['rating'] >= 4) & (df['predicted_sentiment'].isin(['Negativo', 'Neutral']))]

print(f"🔴 FALSOS NEGATIVOS ENCONTRADOS: {len(falsos_negativos)}")
print(f"   ({len(falsos_negativos)/len(df[df['rating'] >= 4])*100:.1f}% de las reseñas con rating 4-5★)")
print(f"   ({len(falsos_negativos)/len(df)*100:.1f}% del total de reseñas)\n")

if len(falsos_negativos) > 0:
    print("--- EJEMPLOS DE FALSOS NEGATIVOS ---")
    for _, row in falsos_negativos.head(8).iterrows():
        print(f"\n⭐ Rating REAL: {row['rating']} → Predicción modelo: {row['predicted_stars']}★ ({row['predicted_sentiment']})")
        print(f"   Confianza: {row['predicted_confidence']:.2%}")
        print(f"   Texto: {row['review_text'][:250]}")

### 11.2 Falsos Positivos (caso menos esperado)

Reseñas con rating bajo (1-2★) pero que el modelo clasifica como positivas.

In [ ]:
# Falsos positivos: rating real BAJO (1-2★) pero modelo dice Positivo
falsos_positivos = df[(df['rating'] <= 2) & (df['predicted_sentiment'] == 'Positivo')]

print(f"🟢 FALSOS POSITIVOS ENCONTRADOS: {len(falsos_positivos)}")
print(f"   ({len(falsos_positivos)/len(df[df['rating'] <= 2])*100:.1f}% de las reseñas con rating 1-2★)\n")

if len(falsos_positivos) > 0:
    print("--- EJEMPLOS DE FALSOS POSITIVOS ---")
    for _, row in falsos_positivos.head(4).iterrows():
        print(f"\n⭐ Rating REAL: {row['rating']} → Predicción modelo: {row['predicted_stars']}★ ({row['predicted_sentiment']})")
        print(f"   Confianza: {row['predicted_confidence']:.2%}")
        print(f"   Texto: {row['review_text'][:250]}")

# Métrica de alineación global
df['coincide'] = df['real_sentiment'] == df['predicted_sentiment']
aciertos = df['coincide'].sum()
print(f"\n{'='*60}")
print(f"📊 MÉTRICA GLOBAL DE ALINEACIÓN")
print(f"{'='*60}")
print(f"Reseñas donde REAL = PREDICHO: {aciertos} / {len(df)} ({aciertos/len(df)*100:.1f}%)")
print(f"Reseñas con discrepancia:     {len(df) - aciertos} / {len(df)} ({(len(df)-aciertos)/len(df)*100:.1f}%)")

---
## 12. Conclusiones para la Account Manager

Respondemos las preguntas clave: ¿el sentimiento escrito coincide con el promedio de 4.5 estrellas? ¿Cuántas reseñas son positivas/neutrales/negativas? ¿De dónde vienen las discrepancias?

In [ ]:
print("=" * 70)
print("📋 RESUMEN EJECUTIVO PARA LA ACCOUNT MANAGER")
print("=" * 70)

print(f"""
🎯 PREGUNTA: ¿El sentimiento expresado en las reseñas escritas coincide con el promedio de 4.5 estrellas?

📊 RESPUESTA: {'PARCIALMENTE — hay discrepancia significativa' if (len(df)-aciertos)/len(df) > 0.1 else 'SÍ — hay alta alineación'}.

{'='*60}
📈 DISTRIBUCIÓN DE SENTIMIENTO (según modelo)
{'='*60}
""")

for sent in ['Positivo', 'Neutral', 'Negativo']:
    n = len(df[df['predicted_sentiment'] == sent])
    print(f"   {sent:12s}: {n:3d} reseñas ({n/len(df)*100:5.1f}%)")

print(f"""
{'='*60}
📊 COMPARATIVA: REAL (rating) vs INFERIDO (modelo)
{'='*60}
""")

for sent in ['Positivo', 'Neutral', 'Negativo']:
    n_real = len(df[df['real_sentiment'] == sent])
    n_pred = len(df[df['predicted_sentiment'] == sent])
    diff = n_pred - n_real
    flecha = '⬆️' if diff > 0 else ('⬇️' if diff < 0 else '➡️')
    print(f"   {sent:12s}: Real={n_real:3d}  →  Modelo={n_pred:3d}  {flecha} ({diff:+d})")

print(f"""
{'='*60}
🔍 DISCREPANCIAS DETECTADAS
{'='*60}

   • Reseñas donde modelo y rating NO coinciden: {len(df)-aciertos} ({(len(df)-aciertos)/len(df)*100:.1f}%)
   • Falsos negativos (rating alto, modelo dice NO positivo): {len(falsos_negativos)} ({len(falsos_negativos)/len(df)*100:.1f}%)
   • Falsos positivos (rating bajo, modelo dice positivo): {len(falsos_positivos)} ({len(falsos_positivos)/len(df)*100:.1f}%)

{'='*60}
💡 CAUSA PRINCIPAL: DESAJUSTE DE DOMINIO
{'='*60}

   El modelo nlptown/bert-base-multilingual-uncased-sentiment fue entrenado con
   reseñas de PRODUCTOS (Amazon-style). Nuestro dataset contiene reseñas de
   SERVICIOS (cafeterías/restaurantes). Esto genera FALSOS NEGATIVOS porque:

   1. El vocabulario de servicio (\"staff\", \"ambiance\", \"service\") no está
      bien representado en el training data del modelo.
   2. Reseñas con lenguaje mixto o matices (ej: \"comida buena pero servicio
      lento\") son clasificadas como neutras/negativas incluso si el rating es 4★.
   3. Palabras como \"slow\", \"wait\" o \"expensive\" activan el clasificador
      negativo aunque el contexto general sea positivo.

{'='*60}
✅ RECOMENDACIÓN A LA ACCOUNT MANAGER
{'='*60}

   1. El promedio de {df['rating'].mean():.1f}★ ES consistente con una mayoría de reseñas
      positivas, pero hay un ~{(len(df)-aciertos)/len(df)*100:.0f}% de reseñas con lenguaje mixto.
   2. Para reportar al cliente: usar la distribución INFERIDA (más conservadora)
      y explicar el desajuste de dominio.
   3. Si se necesita mayor precisión: hacer fine-tuning del modelo con reseñas
      de servicios (dominio específico) para reducir falsos negativos.
""")

print("=" * 70)
print("🏁 FIN DEL ANÁLISIS")
print("=" * 70)